<a href="https://colab.research.google.com/github/Kal-el779/MyPortfolio/blob/main/StellarNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **STELLARNET**- it a model which is used to predict the star types based on the user values.


In [1]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split, cross_val_score

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
!pip install seaborn --quiet

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

IndexError: list index out of range

In [ ]:
# 🧼 STEP 3: Data Cleaning
# Normalize color names
df['Color'] = df['Color'].str.lower().str.strip().str.replace('-', ' ')

In [ ]:
df.info()

In [ ]:
color_map = {
    'blue white': 'blue white',
    'bluewhite': 'blue white',
    'yellowish white': 'yellow white',
    'yellowish': 'yellow',
    'whitish': 'white',
    'white yellow': 'yellow white',
    'pale yellow orange': 'orange',
    'orange red': 'orange',
}
df['Color'] = df['Color'].replace(color_map)

In [ ]:
# Encode Spectral_Class as ordinal: O=0, B=1, ..., M=6
spectral_order = ['O', 'B', 'A', 'F', 'G', 'K', 'M']
df['Spectral_Class'] = pd.Categorical(df['Spectral_Class'], categories=spectral_order, ordered=True)
df['Spectral_Class'] = df['Spectral_Class'].cat.codes

In [ ]:
# Encode Color using LabelEncoder
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Color'] = le.fit_transform(df['Color'])

In [ ]:
le = LabelEncoder()
df['Color_encoded'] = le.fit_transform(df['Color'])
color_labels = le.classes_
sns.countplot(x='Type', hue='Color_encoded', data=df)
plt.legend(title='Color', labels=color_labels)

In [ ]:
features = ['Temperature', 'L', 'R', 'A_M', 'Spectral_Class', 'Color']
X = df[features]
y = df['Type']

In [ ]:
# 🔍 STEP 5: EDA (optional but recommended)
plt.figure(figsize=(8, 6))
sns.heatmap(df[features + [target]].corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# 🎯 STEP 6: Split the Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 🌲 STEP 7: Train Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

In [ ]:
# 📊 STEP 8: Evaluate the Model
y_pred = model.predict(X_test)
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
# 🧪 STEP 9: Cross-Validation Check
scores = cross_val_score(model, X, y, cv=5)
print("\nCross-validation scores:", scores)
print("Mean accuracy:", scores.mean())

In [ ]:
# 🔎 STEP 10: Feature Importance
importances = model.feature_importances_
plt.figure(figsize=(7, 4))
sns.barplot(x=importances, y=features)
plt.title("Feature Importance")
plt.show()

**Adding real star data from GAIA spacecraft, it maps over 1.8 billion stars. **

In [ ]:
!pip install astroquery --quiet

In [ ]:
from astroquery.gaia import Gaia


In [ ]:
Gaia.MAIN_GAIA_TAP_URL = "https://gea.esac.esa.int/tap-server/tap"  # Fallback mirror

In [ ]:
query = """
SELECT TOP 10
    source_id, teff_val, radius_val, lum_val, phot_g_mean_mag
FROM gaiadr3.gaia_source
WHERE teff_val IS NOT NULL
  AND radius_val IS NOT NULL
  AND lum_val IS NOT NULL
  AND phot_g_mean_mag IS NOT NULL
"""

In [ ]:
job = Gaia.launch_job_async(query)
gaia_df = job.get_results().to_pandas()

In [ ]:
gaia_df = gaia_df.rename(columns={
    'teff_val': 'Temperature',
    'radius_val': 'R',
    'lum_val': 'L',
    'phot_g_mean_mag': 'A_M'
})

In [ ]:
# Predict on Gaia data using your trained model
gaia_features = gaia_df[['Temperature', 'L', 'R', 'A_M']]
gaia_df['Predicted_Type'] = model.predict(gaia_features)


In [ ]:
gaia_df[['source_id', 'Temperature', 'L', 'R', 'A_M', 'Predicted_Type']].head()